In [5]:
import pandas as pd
import re
from pathlib import Path


LINE_PATTERN = re.compile(
    r"prompt=(?P<prompt>[^,]+),\s*"
    r"model='(?P<model>[^']+)'\s*"
    r"source='(?P<source>[^']+)',\s*"
    r"target='(?P<target>[^']+)',\s*"
    r"input_tokens=(?P<input_tokens>\d+),\s*"
    r"output_tokens=(?P<output_tokens>\d+),\s*"
    r"proc_time=(?P<proc_time>[0-9.]+)"
)


def read_log_to_df(path: str | Path) -> pd.DataFrame:
    rows = []

    with open(path, "r") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            m = LINE_PATTERN.match(line)
            if not m:
                raise ValueError(f"Failed to parse line {line_no}:\n{line}")

            row = m.groupdict()
            row["input_tokens"] = int(row["input_tokens"])
            row["output_tokens"] = int(row["output_tokens"])
            row["proc_time"] = float(row["proc_time"])
            rows.append(row)

    return pd.DataFrame(rows)


In [6]:
path = Path("../../llm_responses_debug.txt")
path.exists()

True

In [7]:
df = read_log_to_df(path)


def add_derived_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["total_tokens"] = df["input_tokens"] + df["output_tokens"]
    df["tokens_per_sec"] = df["total_tokens"] / df["proc_time"]
    df["output_input_ratio"] = df["output_tokens"] / df["input_tokens"]
    return df


df = add_derived_metrics(df)


def aggregate(
    df: pd.DataFrame,
    groupby: list[str],
    metrics: list[str],
    agg: str | list[str] = "mean",
    round_to: int = 4,
) -> pd.DataFrame:
    result = df.groupby(groupby)[metrics].agg(agg).reset_index()

    numeric_cols = result.select_dtypes("number").columns
    result[numeric_cols] = result[numeric_cols].round(round_to)

    return result


In [8]:
aggregate(
    df,
    groupby=["model", "prompt"],
    metrics=["proc_time", "input_tokens", "output_tokens", "total_tokens"]
)


,model,prompt,proc_time,input_tokens,output_tokens,total_tokens
0,gemini-2.0-flash,prompt_direct_entity,6.8812,99.7992,9.0000,108.7992
1,gemini-2.0-flash,prompt_direct_entity_ontological,5.8949,133.3150,9.0000,142.3150
2,gemini-2.0-flash,prompt_direct_entity_with_synonyms,3.9191,124.0157,9.0000,133.0157
3,gemini-2.0-flash,prompt_sequential_hierarchy,6.7674,119.6732,9.0000,128.6732
4,gemini-2.0-flash,prompt_sequential_hierarchy_ontological,6.9233,173.4567,9.0000,182.4567
5,gemini-2.0-flash,prompt_sequential_hierarchy_with_synonyms,5.6018,189.9134,9.0000,198.9134
6,gemini-2.0-flash-lite,prompt_direct_entity,5.4699,99.7992,9.0000,108.7992
7,gemini-2.0-flash-lite,prompt_direct_entity_ontological,7.0068,133.3150,9.0000,142.3150
8,gemini-2.0-flash-lite,prompt_direct_entity_with_synonyms,6.2617,124.0157,9.0000,133.0157
9,gemini-2.0-flash-lite,prompt_sequential_hierarchy,7.1191,119.6732,9.0000,128.6732
